<a href="https://www.kaggle.com/code/deepshrestha/datamining-abstract-embedding?scriptVersionId=343974727" target="_blank"><img align="left" alt="Kaggle" title="Open in Kaggle" src="https://kaggle.com/static/images/open-in-kaggle.svg"></a>

In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

In [ ]:
df=pd.read_csv("/kaggle/input/datasets/sumitm004/arxiv-scientific-research-papers-dataset/arXiv_scientific_dataset.csv")
df.head()

In [ ]:
df.columns

In [ ]:
df["id"]

In [ ]:
df=df[['id', 'title', 'category','summary']]

In [ ]:
df.head()

In [ ]:
df_test=df[:10]

In [ ]:
df_test.shape

In [ ]:
pip install sentence-transformers

In [ ]:
from sentence_transformers import SentenceTransformer, util

# 1. Load a well-calibrated sentence transformer model
model = SentenceTransformer('all-mpnet-base-v2') 
# Alternative for scientific text: SentenceTransformer('BAAI/bge-base-en-v1.5')
a="""
Graph neural networks have emerged as powerful models for
protein function prediction. We propose a self-supervised
learning framework that uses graph representations of proteins
to capture structural and functional relationships.
"""
b="""Graph Neural Networks (GNNs) have demonstrated significant 
potential for protein function prediction by effectively modeling 
complex biological relationships. In this work, we introduce a 
self-supervised learning framework that represents proteins as graphs, 
enabling the model to learn meaningful structural and functional patterns 
from protein representations.
"""
# 2. Encode abstracts into normalized embeddings (returns PyTorch Tensors by default)
emb0 = model.encode(a, convert_to_tensor=True)
# emb1 = model.encode(b, convert_to_tensor=True)
emb1 = model.encode(df_test["summary"].iloc[1], convert_to_tensor=True)

# 3. Compute Cosine Similarity
similarity = util.cos_sim(emb0, emb1)
print("Cosine Similarity:", similarity.item())

In [ ]:
df.shape

In [ ]:
df

In [ ]:
import gc
import numpy as np
import pandas as pd
import pyarrow as pa
import pyarrow.parquet as pq

output_file = "arxiv_metadata_with_embeddings.parquet"
chunk_size = 20000
num_rows = len(df)
writer = None

# Start multi-process pool to use both GPUs simultaneously
pool = model.start_multi_process_pool()

for start_idx in range(0, num_rows, chunk_size):
    end_idx = min(start_idx + chunk_size, num_rows)
    
    # 1. Slice current chunk
    df_chunk = df.iloc[start_idx:end_idx].copy()
    
    # 2. Encode current chunk across dual GPUs
    chunk_embeddings = model.encode(
        df_chunk["summary"].tolist(),
        pool=pool,
        batch_size=64,
        show_progress_bar=True,
        convert_to_numpy=True
    ).astype(np.float16)
    
    # 3. Attach embedding column to the small chunk
    df_chunk["embedding"] = list(chunk_embeddings)
    
    # 4. Stream directly to Parquet on disk
    table = pa.Table.from_pandas(df_chunk, preserve_index=False)
    if writer is None:
        writer = pq.ParquetWriter(output_file, table.schema)
    writer.write_table(table)
    
    # 5. Force memory release after each batch
    del df_chunk, chunk_embeddings, table
    gc.collect()

# Stop multi-process worker pool
model.stop_multi_process_pool(pool)

if writer:
    writer.close()

print(f"Successfully saved {num_rows} rows with embeddings to {output_file}!")